# NitroGen 2D Platformer Gameplay Training with ConvNeXt-RWKV7

This notebook builds a complete end-to-end pipeline to:
1. **Filter 2D Platformer Games** from [NVIDIA NitroGen](https://huggingface.co/datasets/nvidia/NitroGen) (Mario, Celeste, Hollow Knight, Cuphead, Sonic, Mega Man, Shovel Knight, Dead Cells, etc.).
2. **Download Platformer Gameplay Videos** using the URLs in `metadata.json` via `yt-dlp`.
3. **Extract & Synchronize Video Frames** at the exact annotated timestamps into `/kaggle/working/data/nitrogen_platformer_frames`.
4. **Train the ConvNeXt-RWKV7 Gamepad Model** with DINOv3 pre-trained visual priors on Kaggle **2x T4 GPUs** using PyTorch Lightning DDP with mixed precision.
5. **Run Real-Time Online Recurrent Streaming Inference** on game frames.

### Model Architecture
- **_InputNormalize:** DINOv3 ImageNet mean/std normalization
- **AdaptiveLearnedPool2d:** Adaptive spatial downsampling to 224x224
- **ConvNeXt Backbone:** DINOv3 pre-trained representations (`facebook/dinov3-convnext-tiny-pretrain-lvd1689m`)
- **LearnedWeightedGAP:** Spatial attention pooling + global average pooling
- **CausalConv1d:** Temporal convolution with residual shortcut
- **4x RWKV-7 Blocks:** Linear attention recurrent temporal mixing
- **GamepadHead:** 21-D output (17 button logits via BCE + 4 joystick axes in [-1.0, 1.0] via MSE)


In [ ]:
# Install ConvNeXt Platform, yt-dlp for video downloads, and OpenCV for frame extraction
%pip install -q "git+https://github.com/Gabz4200/ConvNeXt_Platform.git" yt-dlp opencv-python-headless


In [ ]:
import os
import json
import tarfile
import subprocess
from pathlib import Path
from functools import partial

import cv2
import torch
import lightning as L
from huggingface_hub import HfFileSystem
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping, RichProgressBar
from lightning.pytorch.loggers import CSVLogger

# Enable safe checkpoint unpickling on PyTorch 2.6+
os.environ.setdefault("TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD", "1")
L.seed_everything(42, workers=True)

num_gpus = torch.cuda.device_count()
print(f"PyTorch Version: {torch.__version__}")
print(f"Lightning Version: {L.__version__}")
print(f"Available GPUs: {num_gpus}")
for i in range(num_gpus):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")


In [ ]:
# Scan NitroGen shards from Hugging Face and filter for 2D Platformer games
PLATFORMER_KEYWORDS = [
    "mario", "celeste", "hollow knight", "cuphead", "sonic",
    "shovel knight", "rayman", "megaman", "mega man", "dead cells",
    "ori", "castlevania", "metroid", "spelunky", "super meat boy",
    "donkey kong", "crash bandicoot", "kirby", "guacamelee", "inside",
    "limbo", "fez", "broforce", "terraria", "jump king", "pizza tower",
    "platformer", "2d platformer",
]

def is_platformer(game_name: str) -> bool:
    name_lower = game_name.lower()
    return any(kw in name_lower for kw in PLATFORMER_KEYWORDS)

fs = HfFileSystem()
max_shards_to_scan = 3  # Increase to 10-100 on full Kaggle runs
matched_chunks = []

print(f"Scanning first {max_shards_to_scan} NitroGen shards for platformer games...")
for shard_idx in range(max_shards_to_scan):
    shard_path = f"datasets/nvidia/NitroGen/actions/SHARD_{shard_idx:04d}.tar.gz"
    try:
        with fs.open(shard_path, "rb") as f, tarfile.open(fileobj=f, mode="r|gz") as tar:
            for member in tar:
                if member.name.endswith("metadata.json"):
                    extracted = tar.extractfile(member)
                    if extracted:
                        meta = json.load(extracted)
                        game = meta.get("game", "")
                        orig = meta.get("original_video")
                        if is_platformer(game) and orig:
                            matched_chunks.append({
                                "shard_idx": shard_idx,
                                "game": game,
                                "uuid": meta.get("uuid"),
                                "chunk_id": meta.get("chunk_id"),
                                "video_id": orig.get("video_id"),
                                "url": orig.get("url"),
                                "start_time": orig.get("start_time", 0.0),
                                "end_time": orig.get("end_time", 20.0),
                                "start_frame": orig.get("start_frame", 0),
                                "end_frame": orig.get("end_frame", 1200),
                            })
    except (tarfile.TarError, OSError) as err:
        logger.warning("Error reading shard %d: %s", shard_idx, err)

print(f"Found {len(matched_chunks)} platformer chunks across scanned shards!")
for c in matched_chunks[:5]:
    print(f"  Game: {c['game']} | Video ID: {c['video_id']} | URL: {c['url']}")


In [ ]:
# Download gameplay videos and extract synchronized frames
video_frames_dir = Path("/kaggle/working/data/nitrogen_platformer_frames")
video_frames_dir.mkdir(parents=True, exist_ok=True)
downloaded_count = 0
max_videos_to_download = 10  # Set to desired number of platformer videos

for chunk in matched_chunks:
    if downloaded_count >= max_videos_to_download:
        break

    vid_id = chunk["video_id"]
    url = chunk["url"]
    start_frame = chunk["start_frame"]
    end_frame = chunk["end_frame"]
    start_time = chunk["start_time"]
    end_time = chunk["end_time"]

    if not url or not vid_id:
        continue

    frame_dest_dir = video_frames_dir / vid_id
    if (frame_dest_dir / f"frame_{start_frame:06d}.jpg").exists():
        print(f"Frames already extracted for {vid_id}, skipping download.")
        downloaded_count += 1
        continue

    frame_dest_dir.mkdir(parents=True, exist_ok=True)
    temp_video_path = video_frames_dir / f"temp_{vid_id}.mp4"

    # Use yt-dlp to download the 360p/480p stream for fast processing
    ytdlp_cmd = [
        "yt-dlp",
        "-f", "bestvideo[height<=480][ext=mp4]/best[height<=480]/best",
        "--download-sections", f"*{start_time}-{end_time}",
        "--force-keyframes-at-cuts",
        "-o", str(temp_video_path),
        url,
    ]

    try:
        print(f"Downloading clip for {chunk['game']} ({vid_id})...")
        subprocess.run(ytdlp_cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=120)

        # Extract frames using OpenCV
        if temp_video_path.exists():
            cap = cv2.VideoCapture(str(temp_video_path))
            f_idx = 0
            while cap.isOpened():
                ret, frame = cap.read()
                if not ret:
                    break
                abs_idx = start_frame + f_idx
                frame_path = frame_dest_dir / f"frame_{abs_idx:06d}.jpg"
                cv2.imwrite(str(frame_path), frame, [cv2.IMWRITE_JPEG_QUALITY, 85])
                f_idx += 1
            cap.release()
            temp_video_path.unlink(missing_ok=True)
            print(f"Extracted {f_idx} frames for {chunk['game']} ({vid_id})")
            downloaded_count += 1
    except Exception as err:
        print(f"Failed to download {vid_id}: {err}")
        temp_video_path.unlink(missing_ok=True)

print(f"Ready: Downloaded and extracted {downloaded_count} platformer gameplay video clips!")


In [ ]:
from src.data.nitrogen_datamodule import NitroGenDataModule

# Initialize NitroGenDataModule pointing to the extracted real video frames
# The 'Load or Skip' policy cleanly skips chunks whose videos were not downloaded
datamodule = NitroGenDataModule(
    video_dir=str(video_frames_dir),
    batch_size=32,             # Per-GPU batch size (effective 64 on 2x T4)
    max_samples=50000,         # Adjust sample budget
    val_samples=1000,
    steps_per_sample=16,       # 16-step sequence windows for temporal mixing
    single_step=True,          # 1 gamepad state per forward pass (16 samples per window)
    shuffle=True,              # Episode-level shuffle buffer
    shuffle_buffer_size=1000,
    image_size=(224, 224),
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

datamodule.setup("fit")


In [ ]:
from src.models.components.convnext_rwkv7 import ConvNeXtRWKV7Gamepad
from src.models.convnext_rwkv7_module import ConvNeXtRWKV7GamepadLitModule

# 1. Instantiate ConvNeXt-RWKV7 Gamepad model with pre-trained DINOv3 visual priors
net = ConvNeXtRWKV7Gamepad(
    in_chans=3,
    convnext_size="tiny",
    pretrained_dinov3=True,     # Loads facebook/dinov3-convnext-tiny-pretrain-lvd1689m
    freeze_convnext=True,       # Freezes ConvNeXt while allowing backprop to AdaptiveLearnedPool2d
    rwkv_dim=256,
    rwkv_head_size=64,
    rwkv_layers=4,
    head_hidden_dim=256,
    num_buttons=17,
    num_joysticks=2,
)

# 2. Learning rate scheduler and optimizer factories
max_epochs = 15
optimizer_factory = partial(torch.optim.AdamW, lr=1e-3, weight_decay=0.01)
scheduler_factory = partial(torch.optim.lr_scheduler.CosineAnnealingLR, T_max=max_epochs, eta_min=1e-6)

# 3. LightningModule wrapper with combined BCE (17 buttons) + MSE (4 joystick axes) loss
model = ConvNeXtRWKV7GamepadLitModule(
    net=net,
    optimizer=optimizer_factory,
    scheduler=scheduler_factory,
    joystick_loss_weight=1.0,
)


In [ ]:
# Setup multi-GPU DDP Trainer on Kaggle 2x T4
# References: https://lightning.ai/docs/pytorch/stable/reference/common/notebooks
strategy = "ddp_notebook" if num_gpus > 1 else "auto"
devices = num_gpus if num_gpus > 0 else "auto"
accelerator = "gpu" if num_gpus > 0 else "cpu"
precision = "16-mixed" if num_gpus > 0 else "32-true"

logger = CSVLogger(save_dir="logs", name="nitrogen_platformer_rwkv7")
callbacks = [
    ModelCheckpoint(
        dirpath="checkpoints/nitrogen_platformer",
        filename="platformer-{epoch:02d}-{val/loss:.4f}",
        monitor="val/loss",
        mode="min",
        save_top_k=2,
        save_last=True,
    ),
    EarlyStopping(
        monitor="val/loss",
        patience=4,
        mode="min",
    ),
    RichProgressBar(),
]

trainer = L.Trainer(
    accelerator=accelerator,
    devices=devices,
    strategy=strategy,
    precision=precision,
    max_epochs=max_epochs,
    callbacks=callbacks,
    logger=logger,
    log_every_n_steps=25,
    gradient_clip_val=1.0,
)


In [ ]:
# Train ConvNeXt-RWKV7 on the filtered NitroGen 2D platformer gameplay data
trainer.fit(model=model, datamodule=datamodule)

# Evaluate best checkpoint
trainer.test(model=model, datamodule=datamodule, ckpt_path="best")


In [ ]:
# Real-Time Online Recurrent Streaming Inference Demo (Frame-by-Frame)
model.eval()
device = next(model.parameters()).device

# Initialize recurrent state tuple
state = model.net.init_streaming_state(batch_size=1, device=device)

# Load a test sample
datamodule.setup("test")
test_frame, target_actions = next(iter(datamodule.test_dataloader()))
input_frame = test_frame[0:1].to(device)  # Shape: (1, 3, 224, 224)

# Execute O(1) recurrent step
with torch.no_grad():
    (full_gamepad, buttons_logits, joysticks), state = model.net.step(input_frame, state)

btn_probs = buttons_logits.sigmoid().squeeze(0).cpu().tolist()
joy_axes = joysticks.squeeze(0).cpu().tolist()

print(f"Predicted Full Gamepad Vector (21-D): {full_gamepad.shape}")
print("Predicted Button Probabilities (17-D):", [round(p, 3) for p in btn_probs])
print(f"Predicted Left Stick (X, Y): ({joy_axes[0]:.3f}, {joy_axes[1]:.3f})")
print(f"Predicted Right Stick (X, Y): ({joy_axes[2]:.3f}, {joy_axes[3]:.3f})")
